# 光学测量学习 · 第 1 周数值练习

本 Notebook 用 NumPy 和 Matplotlib 复算网站中的三个入门模型：Fraunhofer 单缝衍射、薄透镜 ABCD 近轴成像、圆孔 Airy 尺度与几何离焦圈。

这些是理想教学模型，不是完整镜头处方追迹、矢量高 NA 仿真或特定光刻机模型。

## 1. 单缝 Fraunhofer 衍射

`I/I0 = (sin(beta)/beta)^2`，`beta = pi a sin(theta)/lambda`。屏幕坐标 `x` 与角度关系为 `theta = atan(x/L)`。第一暗纹满足 `a sin(theta) = lambda`；小角度时 `x1 ≈ lambda L / a`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

wavelength_nm = 550.0
slit_width_um = 50.0
distance_m = 1.2

wavelength_m = wavelength_nm * 1e-9
slit_width_m = slit_width_um * 1e-6
theta_first = np.arcsin(wavelength_m / slit_width_m)
first_minimum_mm = distance_m * np.tan(theta_first) * 1e3
small_angle_mm = wavelength_m * distance_m / slit_width_m * 1e3
x_mm = np.linspace(-3 * first_minimum_mm, 3 * first_minimum_mm, 801)
theta = np.arctan((x_mm * 1e-3) / distance_m)
beta = np.pi * slit_width_m * np.sin(theta) / wavelength_m
intensity = np.sinc(beta / np.pi) ** 2  # NumPy's sinc(z) = sin(pi*z)/(pi*z)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_mm, intensity, color='#ca6845', lw=2)
ax.axvline(first_minimum_mm, color='#637c71', ls='--', lw=1)
ax.axvline(-first_minimum_mm, color='#637c71', ls='--', lw=1)
ax.set(xlabel='Screen coordinate x (mm)', ylabel='Normalized intensity I / I0', ylim=(-0.03, 1.05))
ax.grid(alpha=0.2)
plt.show()
print(f'Exact first minimum: {first_minimum_mm:.6f} mm')
print(f'Small-angle estimate: {small_angle_mm:.6f} mm')

## 2. ABCD 近轴薄透镜

光线向量取 `[y, theta]`。传播矩阵 `T(d) = [[1, d], [0, 1]]`，薄透镜矩阵 `L(f) = [[1, 0], [-1/f, 1]]`。系统从物面到像面为 `M = T(v) L(f) T(u)`，矩阵按从右向左作用。成像条件 `B = 0` 导出薄透镜公式 `1/f = 1/u + 1/v`。

In [ ]:
f_mm = 50.0
u_mm = 150.0
v_mm = 1.0 / (1.0 / f_mm - 1.0 / u_mm)
T_u = np.array([[1.0, u_mm], [0.0, 1.0]])
L = np.array([[1.0, 0.0], [-1.0 / f_mm, 1.0]])
T_v = np.array([[1.0, v_mm], [0.0, 1.0]])
M = T_v @ L @ T_u
magnification = -v_mm / u_mm

print(f'Image distance v = {v_mm:.6f} mm')
print(f'Lateral magnification m = {magnification:.6f}')
print('ABCD matrix =')
print(M)
assert abs(M[0, 1]) < 1e-10, 'At the conjugate image plane, B should be zero.'
assert abs(M[0, 0] - magnification) < 1e-10, 'A should equal the lateral magnification.'

## 3. 圆孔衍射与离焦几何弥散圈

理想圆孔在焦面上的 Airy 首暗环半径近似为 `r = 1.22 lambda f / D`。平行入射的近轴光锥若在焦面前后偏移 `delta_z`，几何弥散圈直径近似为 `d = D |delta_z| / f`。两者是不同机制的独立估算；这里不把它们合成为精确的离焦 PSF。

In [ ]:
aperture_mm = 10.0
focal_length_mm = 50.0
wavelength_nm = 550.0
defocus_mm = 1.2
airy_radius_um = 1.22 * wavelength_nm * 1e-3 * focal_length_mm / aperture_mm
defocus_diameter_um = aperture_mm * abs(defocus_mm) / focal_length_mm * 1e3

print(f'Airy first-minimum radius at focus = {airy_radius_um:.3f} um')
print(f'Geometric defocus-circle diameter = {defocus_diameter_um:.3f} um')

## 阅读结论

- 单缝缝宽减小，衍射主峰变宽。
- ABCD 的 `B = 0` 是一阶成像条件；薄透镜模型不包含高阶像差。
- 圆孔有限口径导致理想焦面点像也有衍射尺度。离焦几何圈随 `D` 和 `|delta_z|` 增大、随 `f` 增大而减小。
- 噪声、采样、ROI 和定位算法应与光学像差分层讨论。

## 逐字稿课堂复现（2026-10-28）
对应原稿 986–1085 行。下面恢复课堂代码，复算 v=75、80、90 mm 与 u=100 mm 的对照。只计算一阶近轴几何光线，未实现衍射和高阶像差。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 单位：mm；角度：rad
f, u, v = 50.0, 150.0, 75.0

def free_space(distance):
    return np.array([[1.0, distance], [0.0, 1.0]])

def thin_lens(focal_length):
    return np.array([[1.0, 0.0], [-1.0 / focal_length, 1.0]])

M = free_space(v) @ thin_lens(f) @ free_space(u)
print("系统矩阵：", M, sep="\n")
print("B =", M[0, 1])
for theta in np.linspace(-0.02, 0.02, 5):
    ray_out = M @ np.array([0.0, theta])
    print(f"入射角={theta:.3f} rad, 像面高度={ray_out[0]:.6f} mm")


In [ ]:
# 原稿的两个小实验，加上交互演示的 v=90 mm
for object_distance, sensor_distance in [(150.0,75.0),(150.0,80.0),(150.0,90.0),(100.0,100.0)]:
    matrix = free_space(sensor_distance) @ thin_lens(50.0) @ free_space(object_distance)
    ideal_image = 50.0*object_distance/(object_distance-50.0)
    diameter_um = 8.0*abs(sensor_distance-ideal_image)/ideal_image*1000.0
    print(f"u={object_distance:g}, v={sensor_distance:g} mm; B={matrix[0,1]:g} mm; D=8 mm 几何圈直径={diameter_um:g} μm")

# 第三组：B=-30 mm，几何圈直径1600 μm；光线图用一致的mm坐标。
u, f, v, aperture = 150.0, 50.0, 90.0, 8.0
fig, ax = plt.subplots(figsize=(8, 3))
for lens_height in np.linspace(-aperture/2, aperture/2, 9):
    theta_in = lens_height/u
    theta_out = theta_in-lens_height/f
    y_sensor = lens_height+theta_out*v
    ax.plot([-u,0,v], [0,lens_height,y_sensor], linewidth=1)
ax.axvline(0, color='gray', label='lens / aperture')
ax.axvline(75, color='green', linestyle='--', label='ideal image')
ax.axvline(v, color='red', label='sensor')
ax.set(xlabel='z (mm)', ylabel='y (mm)', title='Paraxial geometry only; no diffraction / aberration')
ax.legend()
plt.show()
